# BERT Emotion Classification Training

**AI Learning Assistant** — Fine-tune `bert-base-uncased` for 5-class student emotion detection.

- **Input text:** original `text` column (BERT handles its own tokenization)
- **Labels:** encoded integers from preprocessing (`label` column)

**Outputs:**
- `models/bert_emotion/` (model + tokenizer)
- `models/bert_label_mapping.json`
- `models/bert_training_history.json`

## 1. Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.bert_model import *
from utils.constants import *

## 2. Load Datasets

We use the **original text** column because BERT's WordPiece tokenizer preserves punctuation, casing context, and subword structure that our BiLSTM preprocessing removes.

In [ ]:
train_df, val_df, test_df, label_mapping = prepare_bert_training_data()
print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
label_mapping

## 3. Tokenize with BertTokenizer

Truncation and padding prepare fixed-length tensors for the Transformer.

In [ ]:
tokenizer = load_bert_tokenizer()
train_dataset, val_dataset, test_dataset = create_emotion_datasets(
    train_df, val_df, test_df, tokenizer
)

sample = tokenizer(train_df["text"].iloc[0], truncation=True, max_length=BERT_MAX_LENGTH)
print("Token IDs:", sample["input_ids"][:15], "...")
print("Tokens:", tokenizer.convert_ids_to_tokens(sample["input_ids"][:15]))

## 4. Build Sequence Classification Model

`BertForSequenceClassification` adds a linear head on top of `[CLS]` for 5 emotion classes.

In [ ]:
model = create_bert_classifier(label_mapping=label_mapping)
print(f"Parameters: {model.num_parameters():,}")
print(f"Device: {get_device()}")

## 5. Fine-Tune BERT

- **Optimizer:** AdamW (Hugging Face Trainer default)
- **Scheduler:** Linear warmup + decay
- **Early stopping:** patience on validation loss

In [ ]:
training_args = build_training_arguments()
trainer, train_result = train_bert_model(
    model, train_dataset, val_dataset, training_args
)
train_result

## 6. Evaluate on Test Set

In [ ]:
eval_results = evaluate_bert_on_test(trainer, test_dataset, label_mapping)
test_metrics = trainer.evaluate(test_dataset)
eval_results["test_loss"] = test_metrics["eval_loss"]

history = extract_epoch_metrics(trainer)
print_bert_report(trainer, train_dataset, eval_results, history)

## 7. Plot Training Curves & Confusion Matrix

In [ ]:
plot_bert_training_history(history, BERT_TRAINING_CURVES_PATH, show=True)

class_names = [label_mapping["id2label"][str(i)] for i in range(NUM_CLASSES)]
plot_bert_confusion_matrix(
    eval_results["confusion_matrix"], class_names, BERT_CONFUSION_MATRIX_PATH, show=True
)

## 8. Save Artifacts

In [ ]:
save_bert_model(trainer.model, tokenizer, BERT_MODEL_DIR)
save_label_mapping(label_mapping, BERT_LABEL_MAPPING_PATH)

history["test_accuracy"] = eval_results["test_accuracy"]
history["test_f1"] = eval_results["test_f1"]
save_training_history(history, BERT_HISTORY_PATH)

print(f"Model: {BERT_MODEL_DIR}")
print(f"Mapping: {BERT_LABEL_MAPPING_PATH}")
print(f"History: {BERT_HISTORY_PATH}")

## 9. Inference Helpers (for later integration)

In [ ]:
sample_texts = train_df["text"].head(3).tolist()
predictions = predict_bert(sample_texts)
probabilities = predict_bert_probabilities(sample_texts)

for text, pred, prob in zip(sample_texts, predictions, probabilities):
    print(f"Text: {text[:80]}...")
    print(f"  Predicted: {pred} (confidence: {prob.max():.2f})\n")